# 08a — Headline Results & Comparison Tables (K-Fold Aggregated)
## Multi-Attribute Scene Classification on nuScenes Front-Camera Images

**This notebook produces the headline numbers and ranking tables for the report's Results section.** It aggregates the per-fit metrics from notebook 06 across 5 folds, computing **mean ± std macro-F1** per (model × version × attribute).

### Tables produced

| Table | Purpose |
|---|---|
| **Headline** | Best tuned model per attribute, mean ± std across folds × seeds |
| **All-results matrix** | One row per model × version, columns = attributes |
| **Cross-attribute ranking** | Which model is most consistent across attributes? |
| **Top-3 ranking per attribute** | First/Second/Third model per attribute |
| **Recall + Precision** | Per-class precision/recall for best models |
| **Fold-by-fold breakdown** | Macro-F1 per fold for the winning model (transparency) |

### Companion notebooks

- `08b_visual_analysis_and_stats.ipynb` — confusion matrices, ROC curves, statistical tests
- `08c_deep_analysis.ipynb` — feature importance, error analysis, RQ summary + scaling discussion


## 0. Setup

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

DATASET_VERSION = 'v1.0-mini'
RESULTS_DIR = Path('results/metrics')
PRED_DIR    = Path('results/predictions')
SPLIT_DIR   = Path('data/splits')
FINAL_DIR   = Path('results/final')
FIG_DIR     = Path('results/figures/final')
for p in [FINAL_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ATTRIBUTES = ['time_of_day', 'weather', 'vehicle_density', 'vru_present']
CLASS_ORDERS = {
    'time_of_day':     ['day', 'night'],
    'weather':         ['clear', 'rain'],
    'vehicle_density': ['low', 'medium', 'high'],
    'vru_present':     ['absent', 'present'],
}
MODEL_NAMES = ['LogisticRegression', 'SVM_RBF', 'RandomForest', 'XGBoost', 'MLP']
DISPLAY_NAMES = {
    'LogisticRegression': 'LogReg',
    'SVM_RBF':            'SVM',
    'RandomForest':       'RF',
    'XGBoost':            'XGB',
    'MLP':                'MLP',
}

print(f'DATASET_VERSION = {DATASET_VERSION}')

## 1. Load Inputs

In [ ]:
df_metrics  = pd.read_csv(RESULTS_DIR / 'all_metrics.csv')
df_baselines = pd.read_csv(RESULTS_DIR / 'baseline_metrics.csv')

# Skipped combos (some (fold, attr) skipped due to single-class train)
skipped_combos = []
skip_path = RESULTS_DIR / 'skipped_combos.json'
if skip_path.exists():
    with open(skip_path) as f:
        skipped_combos = json.load(f)

print(f'Metrics rows:    {len(df_metrics)}')
print(f'Baseline rows:   {len(df_baselines)}')
print(f'Skipped combos:  {len(skipped_combos)}')

test_only  = df_metrics[df_metrics['split'] == 'test'].copy()
tuned_only = test_only[test_only['version'] == 'tuned'].copy()
print(f'Test-set rows:   {len(test_only)}   (tuned only: {len(tuned_only)})')

# Available folds (might be less than 5 for some attributes due to skips)
folds_per_attr = {}
for attr in ATTRIBUTES:
    folds_per_attr[attr] = sorted(tuned_only[tuned_only['attribute'] == attr]['fold'].unique())
print(f'\nFolds available per attribute (after skips):')
for attr in ATTRIBUTES:
    print(f'  {attr:18s}: {folds_per_attr[attr]}')

## 2. Headline Table — Best Tuned Model per Attribute

For each attribute, identify the tuned model with highest mean test macro-F1 across all folds × seeds.

In [ ]:
headline_rows = []
for attr in ATTRIBUTES:
    sub = tuned_only[tuned_only['attribute'] == attr]
    if sub.empty:
        headline_rows.append({
            'attribute': attr, 'best_model': 'N/A',
            'accuracy': 'N/A', 'macro_f1': 'N/A',
            'macro_precision': 'N/A', 'macro_recall': 'N/A',
            'n_folds_evaluated': 0,
        })
        continue

    by_model = sub.groupby('model')[['accuracy', 'macro_f1',
                                      'macro_precision', 'macro_recall']].agg(
                                      ['mean', 'std']).round(3)
    winner = by_model[('macro_f1', 'mean')].idxmax()
    winner_row = by_model.loc[winner]
    n_folds = len(folds_per_attr[attr])
    headline_rows.append({
        'attribute': attr,
        'best_model': DISPLAY_NAMES[winner],
        'accuracy':       f'{winner_row[("accuracy", "mean")]:.3f} ± {winner_row[("accuracy", "std")]:.3f}',
        'macro_f1':       f'{winner_row[("macro_f1", "mean")]:.3f} ± {winner_row[("macro_f1", "std")]:.3f}',
        'macro_precision': f'{winner_row[("macro_precision", "mean")]:.3f} ± {winner_row[("macro_precision", "std")]:.3f}',
        'macro_recall':    f'{winner_row[("macro_recall", "mean")]:.3f} ± {winner_row[("macro_recall", "std")]:.3f}',
        'n_folds_evaluated': n_folds,
    })

df_headline = pd.DataFrame(headline_rows)
print('HEADLINE TABLE — best tuned model per attribute (k-fold aggregated):')
display(df_headline)
df_headline.to_csv(FINAL_DIR / 'headline_table.csv', index=False)
print(f'\nSaved → {FINAL_DIR / "headline_table.csv"}')

## 3. All-Results Matrix (k-fold aggregated)

One row per model × version, columns = attributes. Mirrors Table 7 from the past pneumonia A-grade report.

In [ ]:
matrix_rows = []
for model_name in MODEL_NAMES:
    for version in ['base', 'tuned']:
        row = {'Model': DISPLAY_NAMES[model_name], 'Version': version}
        for attr in ATTRIBUTES:
            sub = test_only[(test_only['model'] == model_name) &
                            (test_only['version'] == version) &
                            (test_only['attribute'] == attr)]
            if sub.empty:
                row[attr] = 'N/A'
            else:
                row[attr] = round(sub['macro_f1'].mean(), 3)
        matrix_rows.append(row)

df_matrix = pd.DataFrame(matrix_rows)

# Add baseline reference rows
baseline_rows = []
for baseline_name in ['random', 'majority_class']:
    row = {'Model': baseline_name.replace('_', '-'), 'Version': 'baseline'}
    for attr in ATTRIBUTES:
        sub = df_baselines[(df_baselines['baseline'] == baseline_name) &
                            (df_baselines['attribute'] == attr)]
        if sub.empty:
            row[attr] = 'N/A'
        else:
            row[attr] = round(sub['macro_f1'].mean(), 3)
    baseline_rows.append(row)
df_matrix = pd.concat([pd.DataFrame(baseline_rows), df_matrix], ignore_index=True)

print('ALL-RESULTS MATRIX — test macro-F1 (mean across folds × seeds):')
display(df_matrix)
df_matrix.to_csv(FINAL_DIR / 'all_results_matrix.csv', index=False)
print(f'\nSaved → {FINAL_DIR / "all_results_matrix.csv"}')

In [ ]:
# Visualise as heatmap (numeric rows only)
numeric_df = df_matrix.copy()
for attr in ATTRIBUTES:
    numeric_df[attr] = pd.to_numeric(numeric_df[attr], errors='coerce')

heat_data = numeric_df.set_index(['Model', 'Version'])[ATTRIBUTES]
fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(heat_data, annot=True, fmt='.3f', cmap='YlGn', vmin=0.3, vmax=1.0,
            ax=ax, cbar_kws={'label': 'Test macro-F1 (k-fold aggregated)'},
            linewidths=0.5)
ax.set_title(f'All models × all attributes — test macro-F1 (k-fold) — {DATASET_VERSION}',
              fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'all_results_matrix_heatmap.png', bbox_inches='tight')
plt.show()

## 4. Cross-Attribute Model Ranking

Which tuned model is most consistent across all four attributes?

In [ ]:
ranking = (tuned_only.groupby('model')['macro_f1']
                       .agg(['mean', 'std', 'min', 'max']).round(3))
ranking = ranking.reindex(MODEL_NAMES)
ranking['mean_rank'] = ranking['mean'].rank(ascending=False).astype(int)
ranking = ranking.sort_values('mean_rank')
ranking.index = [DISPLAY_NAMES[m] for m in ranking.index]
ranking.index.name = 'Model'

print('CROSS-ATTRIBUTE RANKING (tuned, averaged across attributes × folds × seeds):')
display(ranking)
ranking.to_csv(FINAL_DIR / 'cross_attribute_ranking.csv')
print(f'\nSaved → {FINAL_DIR / "cross_attribute_ranking.csv"}')

## 5. Top-3 Ranking Per Attribute

In [ ]:
top3_rows = []
for attr in ATTRIBUTES:
    sub = tuned_only[tuned_only['attribute'] == attr]
    if sub.empty: continue
    by_model = sub.groupby('model')['macro_f1'].mean().round(3).sort_values(ascending=False)
    for rank, (model_name, score) in enumerate(by_model.head(3).items(), 1):
        top3_rows.append({
            'attribute': attr,
            'rank': ['First', 'Second', 'Third'][rank - 1],
            'model': DISPLAY_NAMES[model_name],
            'macro_f1': score,
        })

df_top3 = pd.DataFrame(top3_rows)
df_top3.to_csv(FINAL_DIR / 'top3_per_attribute.csv', index=False)

# Clean display format
display_rows = []
for attr in ATTRIBUTES:
    sub = df_top3[df_top3['attribute'] == attr].sort_values('rank',
        key=lambda x: x.map({'First': 1, 'Second': 2, 'Third': 3}))
    row = {'attribute': attr}
    for r in ['First', 'Second', 'Third']:
        rsub = sub[sub['rank'] == r]
        if not rsub.empty:
            m = rsub.iloc[0]['model']
            f = rsub.iloc[0]['macro_f1']
            row[r] = f'{m} ({f:.3f})'
    display_rows.append(row)
df_top3_clean = pd.DataFrame(display_rows)
print('TOP-3 (clean format):')
display(df_top3_clean)
print(f'\nSaved → {FINAL_DIR / "top3_per_attribute.csv"}')

## 6. Recall + Precision for Best Models (Per Class)

In [ ]:
from sklearn.metrics import precision_score, recall_score

df_preds = pd.read_csv(PRED_DIR / 'predictions_test.csv')

rp_rows = []
for attr in ATTRIBUTES:
    sub = tuned_only[tuned_only['attribute'] == attr]
    if sub.empty: continue
    best_model = sub.groupby('model')['macro_f1'].mean().idxmax()

    # Aggregate predictions across all folds × seeds for this best model
    pred_sub = df_preds[(df_preds['attribute'] == attr) &
                        (df_preds['model'] == best_model) &
                        (df_preds['version'] == 'tuned')]

    # Compute per-fold per-seed precision/recall and average
    per_run = []
    for (fold, seed), grp in pred_sub.groupby(['fold', 'seed']):
        yt = grp['true_label'].values
        yp = grp['pred_label'].values
        num_class = len(CLASS_ORDERS[attr])
        labels = list(range(num_class))
        prec = precision_score(yt, yp, average=None, labels=labels, zero_division=0)
        rec  = recall_score(yt, yp, average=None, labels=labels, zero_division=0)
        per_run.append({'precision': prec, 'recall': rec})

    mean_prec = np.mean([r['precision'] for r in per_run], axis=0)
    mean_rec  = np.mean([r['recall']    for r in per_run], axis=0)

    for cls_i, cls_name in enumerate(CLASS_ORDERS[attr]):
        rp_rows.append({
            'attribute':   attr,
            'best_model':  DISPLAY_NAMES[best_model],
            'class':       cls_name,
            'precision':   round(float(mean_prec[cls_i]), 3),
            'recall':      round(float(mean_rec[cls_i]),  3),
        })

df_rp = pd.DataFrame(rp_rows)
print('RECALL + PRECISION FOR BEST MODELS (per class, mean across folds × seeds):')
display(df_rp)
df_rp.to_csv(FINAL_DIR / 'recall_precision_best.csv', index=False)
print(f'\nSaved → {FINAL_DIR / "recall_precision_best.csv"}')

## 7. Fold-by-Fold Breakdown (transparency)

For each attribute, show macro-F1 per fold for the winning tuned model.

In [ ]:
fold_break_rows = []
for attr in ATTRIBUTES:
    sub = tuned_only[tuned_only['attribute'] == attr]
    if sub.empty: continue
    best_model = sub.groupby('model')['macro_f1'].mean().idxmax()
    msub = sub[sub['model'] == best_model]
    per_fold = msub.groupby('fold')['macro_f1'].agg(['mean', 'std']).round(3)
    for fold_id, row_data in per_fold.iterrows():
        fold_break_rows.append({
            'attribute': attr,
            'best_model': DISPLAY_NAMES[best_model],
            'fold': fold_id,
            'macro_f1_mean': row_data['mean'],
            'macro_f1_std': row_data['std'],
        })

df_fold_break = pd.DataFrame(fold_break_rows)
print('Per-fold breakdown for winning tuned model per attribute:')
display(df_fold_break.pivot(index='attribute', columns='fold', values='macro_f1_mean'))
df_fold_break.to_csv(FINAL_DIR / 'fold_breakdown.csv', index=False)
print(f'\nSaved → {FINAL_DIR / "fold_breakdown.csv"}')

## 8. Headline Summary JSON

In [ ]:
summary = {
    'dataset_version': DATASET_VERSION,
    'methodology': '5-fold scene-aware cross-validation (Demšar 2006)',
    'attributes': ATTRIBUTES,
    'n_skipped_combos': len(skipped_combos),
    'skipped_combos': skipped_combos,
    'best_model_per_attribute': {
        row['attribute']: row['best_model'] for _, row in df_headline.iterrows()
    },
    'best_macro_f1_per_attribute': {
        row['attribute']: row['macro_f1'] for _, row in df_headline.iterrows()
    },
    'most_consistent_model': ranking.index[0],
    'most_consistent_mean_macro_f1': float(ranking.iloc[0]['mean']),
}

with open(FINAL_DIR / 'headline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved → {FINAL_DIR / "headline_summary.json"}')
print(json.dumps(summary, indent=2))

---
## Findings & Decisions (fill in after running)

### Headline numbers (k-fold aggregated)

- `time_of_day`:     best tuned = **_model_** with macro-F1 = **_X.XXX ± Y.YYY_**
- `weather`:         best tuned = **_model_** with macro-F1 = **_X.XXX ± Y.YYY_** (across _N_ folds)
- `vehicle_density`: best tuned = **_model_** with macro-F1 = **_X.XXX ± Y.YYY_**
- `vru_present`:     best tuned = **_model_** with macro-F1 = **_X.XXX ± Y.YYY_**

### Most consistent model

- **_model_** with mean macro-F1 = **_X.XXX_** across 4 attributes

### v1.0-mini limitations acknowledged

- _N_ (fold, attribute) combinations were skipped due to single-class training sets
- Specifically: `weather` may be evaluated in only 1-4 folds depending on rain scene distribution

### Next analyses

- See `08b` for confusion matrices, ROC, and statistical tests
- See `08c` for feature importance, error analysis, and scaling-to-trainval roadmap
